In [1]:
# Cell 1: Install and import libraries
import os
from pathlib import Path
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.windows import from_bounds
import numpy as np
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from shapely.geometry import box

import rasterio
from rasterio.mask import mask
from shapely.geometry import box
from pathlib import Path
import os


In [2]:
# Reference files
reference_file_010 = "READY_data/clipped_landuse_data/clipped_history_2020.tif"
reference_file_025 = "READY_data/clipped_landuse_data_025/clipped_history_2020_025.tif"

# Folder to check
input_folder = "READY_data/inputs"

print("Checking file alignment in inputs folder\n")
print("=" * 70)

# Load reference properties
print("REFERENCE FILES:\n")

with rasterio.open(reference_file_010) as src:
    reference_010 = {
        'bounds': src.bounds,
        'transform': src.transform,
        'shape': (src.height, src.width),
        'crs': src.crs,
        'resolution': src.res
    }
    print(f"0.1° reference: {Path(reference_file_010).name}")
    print(f"  Shape: {reference_010['shape']}")
    print(f"  Bounds: ({reference_010['bounds'].left:.2f}, {reference_010['bounds'].bottom:.2f}) to ({reference_010['bounds'].right:.2f}, {reference_010['bounds'].top:.2f})")
    print(f"  Resolution: {reference_010['resolution']}")
    print(f"  CRS: {reference_010['crs']}")

print()

with rasterio.open(reference_file_025) as src:
    reference_025 = {
        'bounds': src.bounds,
        'transform': src.transform,
        'shape': (src.height, src.width),
        'crs': src.crs,
        'resolution': src.res
    }
    print(f"0.25° reference: {Path(reference_file_025).name}")
    print(f"  Shape: {reference_025['shape']}")
    print(f"  Bounds: ({reference_025['bounds'].left:.2f}, {reference_025['bounds'].bottom:.2f}) to ({reference_025['bounds'].right:.2f}, {reference_025['bounds'].top:.2f})")
    print(f"  Resolution: {reference_025['resolution']}")
    print(f"  CRS: {reference_025['crs']}")

print("\n" + "=" * 70)
print(f"\nCHECKING FILES IN {input_folder}:\n")

# Get all files from inputs folder
input_files = list(Path(input_folder).glob("*.tif")) + list(Path(input_folder).glob("*.tiff"))
print(f"Found {len(input_files)} files\n")

all_aligned = True

for geotiff_file in sorted(input_files):
    # Determine if file is 0.1° or 0.25° based on filename
    is_025 = geotiff_file.stem.endswith('_025') or geotiff_file.stem.endswith('025')
    is_010 = geotiff_file.stem.endswith('_010') or geotiff_file.stem.endswith('010')
    
    if is_025:
        reference = reference_025
        res_label = "0.25"
    elif is_010:
        reference = reference_010
        res_label = "0.1"
    else:
        print(f"{geotiff_file.name} - Cannot determine resolution from filename")
        continue
    
    with rasterio.open(geotiff_file) as src:
        props = {
            'bounds': src.bounds,
            'transform': src.transform,
            'shape': (src.height, src.width),
            'crs': src.crs,
            'resolution': src.res
        }
        
        misaligned = False
        issues = []
        
        if props['shape'] != reference['shape']:
            issues.append(f"Shape: {props['shape']} vs {reference['shape']}")
            misaligned = True
            all_aligned = False
        
        if props['bounds'] != reference['bounds']:
            issues.append(f"Bounds: ({props['bounds'].left:.2f}, {props['bounds'].bottom:.2f}) to ({props['bounds'].right:.2f}, {props['bounds'].top:.2f})")
            issues.append(f"  Expected: ({reference['bounds'].left:.2f}, {reference['bounds'].bottom:.2f}) to ({reference['bounds'].right:.2f}, {reference['bounds'].top:.2f})")
            misaligned = True
            all_aligned = False
        
        if props['resolution'] != reference['resolution']:
            issues.append(f"Resolution: {props['resolution']} vs {reference['resolution']}")
            misaligned = True
            all_aligned = False
        
        if props['crs'] != reference['crs']:
            issues.append(f"CRS: {props['crs']} vs {reference['crs']}")
            misaligned = True
            all_aligned = False
        
        if misaligned:
            print(f"✗ {geotiff_file.name} ({res_label})")
            for issue in issues:
                print(f"  {issue}")
            print()
        else:
            print(f"✓ {geotiff_file.name} ({res_label})")

print("=" * 70)
if all_aligned:
    print("ALL FILES ARE PROPERLY ALIGNED!")
else:
    print("SOME FILES ARE NOT ALIGNED - see details above")

Checking file alignment in inputs folder

REFERENCE FILES:

0.1° reference: clipped_history_2020.tif
  Shape: (485, 570)
  Bounds: (-25.00, 32.41) to (32.00, 80.91)
  Resolution: (0.1, 0.1)
  CRS: EPSG:4326

0.25° reference: clipped_history_2020_025.tif
  Shape: (194, 228)
  Bounds: (-25.00, 32.41) to (32.00, 80.91)
  Resolution: (0.25, 0.25)
  CRS: EPSG:4326


CHECKING FILES IN READY_data/inputs:

Found 4 files

✓ 2019_gdp_aligned_010.tif (0.1)
✓ 2019_gdp_aligned_025.tif (0.25)
✓ 2020_pop_aligned_010.tif (0.1)
✓ 2020_pop_aligned_025.tif (0.25)
ALL FILES ARE PROPERLY ALIGNED!


# Check label alingment

In [4]:
# Cell: Check alignment of CISI label files
labels_folder = "READY_data/labels"

print("Checking CISI label files alignment\n")
print("=" * 70)

# Reference files
reference_file_010 = "READY_data/clipped_landuse_data/clipped_history_2020.tif"
reference_file_025 = "READY_data/clipped_landuse_data_025/clipped_history_2020_025.tif"

# Load reference properties
with rasterio.open(reference_file_010) as src:
    reference_010 = {
        'bounds': src.bounds,
        'shape': (src.height, src.width),
        'crs': src.crs,
        'resolution': src.res
    }
    print(f"0.1° REFERENCE: {Path(reference_file_010).name}")
    print(f"  Shape: {reference_010['shape']}")
    print(f"  Bounds: ({reference_010['bounds'].left:.2f}, {reference_010['bounds'].bottom:.2f}) to ({reference_010['bounds'].right:.2f}, {reference_010['bounds'].top:.2f})")
    print(f"  Resolution: {reference_010['resolution']}")
    print(f"  CRS: {reference_010['crs']}")

print()

with rasterio.open(reference_file_025) as src:
    reference_025 = {
        'bounds': src.bounds,
        'shape': (src.height, src.width),
        'crs': src.crs,
        'resolution': src.res
    }
    print(f"0.25° REFERENCE: {Path(reference_file_025).name}")
    print(f"  Shape: {reference_025['shape']}")
    print(f"  Bounds: ({reference_025['bounds'].left:.2f}, {reference_025['bounds'].bottom:.2f}) to ({reference_025['bounds'].right:.2f}, {reference_025['bounds'].top:.2f})")
    print(f"  Resolution: {reference_025['resolution']}")
    print(f"  CRS: {reference_025['crs']}")

print("\n" + "=" * 70)
print(f"\nCHECKING LABEL FILES IN {labels_folder}:\n")

# Get all label files
label_files = list(Path(labels_folder).glob("*.tif")) + list(Path(labels_folder).glob("*.tiff"))
print(f"Found {len(label_files)} label files\n")

all_aligned = True

for label_file in sorted(label_files):
    # Determine resolution based on filename
    is_025 = '025' in label_file.stem or '0.25' in label_file.stem or '25' in label_file.stem
    is_010 = '010' in label_file.stem or '0.1' in label_file.stem or '10' in label_file.stem
    
    if is_025:
        reference = reference_025
        res_label = "0.25°"
    elif is_010:
        reference = reference_010
        res_label = "0.1°"
    else:
        print(f"⚠ {label_file.name} - Cannot determine resolution from filename")
        continue
    
    with rasterio.open(label_file) as src:
        props = {
            'bounds': src.bounds,
            'shape': (src.height, src.width),
            'crs': src.crs,
            'resolution': src.res
        }
        
        misaligned = False
        issues = []
        
        if props['shape'] != reference['shape']:
            issues.append(f"Shape: {props['shape']} vs {reference['shape']}")
            misaligned = True
            all_aligned = False
        
        if props['bounds'] != reference['bounds']:
            issues.append(f"Bounds: ({props['bounds'].left:.2f}, {props['bounds'].bottom:.2f}) to ({props['bounds'].right:.2f}, {props['bounds'].top:.2f})")
            issues.append(f"  Expected: ({reference['bounds'].left:.2f}, {reference['bounds'].bottom:.2f}) to ({reference['bounds'].right:.2f}, {reference['bounds'].top:.2f})")
            misaligned = True
            all_aligned = False
        
        if props['resolution'] != reference['resolution']:
            issues.append(f"Resolution: {props['resolution']} vs {reference['resolution']}")
            misaligned = True
            all_aligned = False
        
        if props['crs'] != reference['crs']:
            issues.append(f"CRS: {props['crs']} vs {reference['crs']}")
            misaligned = True
            all_aligned = False
        
        if misaligned:
            print(f"✗ {label_file.name} ({res_label})")
            for issue in issues:
                print(f"  {issue}")
            print()
        else:
            print(f"✓ {label_file.name} ({res_label})")

print("=" * 70)
if all_aligned:
    print("✓ ALL LABEL FILES ARE PROPERLY ALIGNED!")
else:
    print("✗ SOME LABEL FILES ARE NOT ALIGNED - see details above")

Checking CISI label files alignment

0.1° REFERENCE: clipped_history_2020.tif
  Shape: (485, 570)
  Bounds: (-25.00, 32.41) to (32.00, 80.91)
  Resolution: (0.1, 0.1)
  CRS: EPSG:4326

0.25° REFERENCE: clipped_history_2020_025.tif
  Shape: (194, 228)
  Bounds: (-25.00, 32.41) to (32.00, 80.91)
  Resolution: (0.25, 0.25)
  CRS: EPSG:4326


CHECKING LABEL FILES IN READY_data/labels:

Found 2 label files

✓ 2024_CISI_010deg.tif (0.1°)
✓ 2024_CISI_025deg.tif (0.25°)
✓ ALL LABEL FILES ARE PROPERLY ALIGNED!


# Final checks on all file alignment and transform alignment

In [6]:
# Cell: Final comprehensive alignment check
print("=" * 70)
print("FINAL COMPREHENSIVE ALIGNMENT CHECK")
print("=" * 70)

# Define all folders and files to check
folders_to_check = {
    "Landuse 0.1°": "READY_data/clipped_landuse_data",
    "Landuse 0.25°": "READY_data/clipped_landuse_data_025",
    "Features (inputs)": "READY_data/inputs",
    "Labels (CISI)": "READY_data/labels"
}

# Reference files
reference_010 = "READY_data/clipped_landuse_data/clipped_history_2020.tif"
reference_025 = "READY_data/clipped_landuse_data_025/clipped_history_2020_025.tif"

# Load reference properties
print("\nREFERENCE GRIDS:\n")
with rasterio.open(reference_010) as src:
    ref_010 = {
        'bounds': src.bounds,
        'shape': (src.height, src.width),
        'crs': src.crs,
        'resolution': src.res
    }
    print(f"0.1° Grid:")
    print(f"  Shape: {ref_010['shape']}")
    print(f"  Bounds: ({ref_010['bounds'].left:.2f}, {ref_010['bounds'].bottom:.2f}) to ({ref_010['bounds'].right:.2f}, {ref_010['bounds'].top:.2f})")
    print(f"  Resolution: {ref_010['resolution']}")
    print(f"  CRS: {ref_010['crs']}")

print()

with rasterio.open(reference_025) as src:
    ref_025 = {
        'bounds': src.bounds,
        'shape': (src.height, src.width),
        'crs': src.crs,
        'resolution': src.res
    }
    print(f"0.25° Grid:")
    print(f"  Shape: {ref_025['shape']}")
    print(f"  Bounds: ({ref_025['bounds'].left:.2f}, {ref_025['bounds'].bottom:.2f}) to ({ref_025['bounds'].right:.2f}, {ref_025['bounds'].top:.2f})")
    print(f"  Resolution: {ref_025['resolution']}")
    print(f"  CRS: {ref_025['crs']}")

print("\n" + "=" * 70)

# Check all folders
all_aligned = True
total_files_010 = 0
total_files_025 = 0
aligned_files_010 = 0
aligned_files_025 = 0

for folder_name, folder_path in folders_to_check.items():
    print(f"\n### {folder_name.upper()} ###\n")
    
    files = list(Path(folder_path).glob("*.tif")) + list(Path(folder_path).glob("*.tiff"))
    
    if len(files) == 0:
        print(f"  No files found in {folder_path}")
        continue
    
    for file in sorted(files):
        # Determine resolution
        is_025 = any(x in file.stem.lower() for x in ['025', '0.25', '_25'])
        is_010 = any(x in file.stem.lower() for x in ['010', '0.1', '_10']) or not is_025
        
        if is_025:
            reference = ref_025
            res_label = "0.25°"
            total_files_025 += 1
        else:
            reference = ref_010
            res_label = "0.1°"
            total_files_010 += 1
        
        with rasterio.open(file) as src:
            props = {
                'bounds': src.bounds,
                'shape': (src.height, src.width),
                'crs': src.crs,
                'resolution': src.res
            }
            
            # Check alignment
            is_aligned = (
                props['shape'] == reference['shape'] and
                props['bounds'] == reference['bounds'] and
                props['crs'] == reference['crs'] and
                props['resolution'] == reference['resolution']
            )
            
            if is_aligned:
                print(f"✓ {file.name} ({res_label})")
                if is_025:
                    aligned_files_025 += 1
                else:
                    aligned_files_010 += 1
            else:
                print(f"✗ {file.name} ({res_label})")
                if props['shape'] != reference['shape']:
                    print(f"  Shape: {props['shape']} vs {reference['shape']}")
                if props['bounds'] != reference['bounds']:
                    print(f"  Bounds: ({props['bounds'].left:.2f}, {props['bounds'].bottom:.2f}) to ({props['bounds'].right:.2f}, {props['bounds'].top:.2f})")
                    print(f"    Expected: ({reference['bounds'].left:.2f}, {reference['bounds'].bottom:.2f}) to ({reference['bounds'].right:.2f}, {reference['bounds'].top:.2f})")
                if props['resolution'] != reference['resolution']:
                    print(f"  Resolution: {props['resolution']} vs {reference['resolution']}")
                if props['crs'] != reference['crs']:
                    print(f"  CRS: {props['crs']} vs {reference['crs']}")
                all_aligned = False

print("\n" + "=" * 70)
print("SUMMARY:")
print("=" * 70)
print(f"\n0.1° files: {aligned_files_010}/{total_files_010} aligned")
print(f"0.25° files: {aligned_files_025}/{total_files_025} aligned")
print(f"\nTotal: {aligned_files_010 + aligned_files_025}/{total_files_010 + total_files_025} files aligned")

if all_aligned:
    print("\n🎉 ALL FILES ARE PERFECTLY ALIGNED! 🎉")
else:
    print("\n⚠ SOME FILES ARE NOT ALIGNED - see details above")

FINAL COMPREHENSIVE ALIGNMENT CHECK

REFERENCE GRIDS:

0.1° Grid:
  Shape: (485, 570)
  Bounds: (-25.00, 32.41) to (32.00, 80.91)
  Resolution: (0.1, 0.1)
  CRS: EPSG:4326

0.25° Grid:
  Shape: (194, 228)
  Bounds: (-25.00, 32.41) to (32.00, 80.91)
  Resolution: (0.25, 0.25)
  CRS: EPSG:4326


### LANDUSE 0.1° ###

✓ clipped_history_2020.tif (0.1°)
✓ clipped_ssp1_26_2030.tif (0.1°)
✓ clipped_ssp1_26_2050.tif (0.1°)
✓ clipped_ssp1_26_2100.tif (0.1°)
✓ clipped_ssp2_45_2030.tif (0.1°)
✓ clipped_ssp2_45_2050.tif (0.1°)
✓ clipped_ssp2_45_2100.tif (0.1°)
✓ clipped_ssp3_70_2030.tif (0.1°)
✓ clipped_ssp3_70_2050.tif (0.1°)
✓ clipped_ssp3_70_2100.tif (0.1°)
✓ clipped_ssp4_34_2030.tif (0.1°)
✓ clipped_ssp4_34_2050.tif (0.1°)
✓ clipped_ssp4_34_2100.tif (0.1°)
✓ clipped_ssp5_85_2030.tif (0.1°)
✓ clipped_ssp5_85_2050.tif (0.1°)
✓ clipped_ssp5_85_2100.tif (0.1°)

### LANDUSE 0.25° ###

✓ clipped_history_2020_025.tif (0.25°)
✓ clipped_ssp1_26_2030_025.tif (0.25°)
✓ clipped_ssp1_26_2050_025.tif (0.25°

In [7]:
# Cell: Check pixel-level alignment (transform check)
print("=" * 70)
print("PIXEL-LEVEL ALIGNMENT CHECK")
print("=" * 70)

# Get reference transforms
with rasterio.open(reference_010) as src:
    ref_transform_010 = src.transform
    
with rasterio.open(reference_025) as src:
    ref_transform_025 = src.transform

print("\nReference transforms:")
print(f"0.1°: {ref_transform_010}")
print(f"0.25°: {ref_transform_025}")

print("\n" + "=" * 70)
print("\nChecking if transforms match exactly:\n")

all_transforms_match = True

for folder_name, folder_path in folders_to_check.items():
    print(f"### {folder_name.upper()} ###")
    
    files = list(Path(folder_path).glob("*.tif")) + list(Path(folder_path).glob("*.tiff"))
    
    for file in sorted(files):
        is_025 = any(x in file.stem.lower() for x in ['025', '0.25', '_25'])
        ref_transform = ref_transform_025 if is_025 else ref_transform_010
        res_label = "0.25°" if is_025 else "0.1°"
        
        with rasterio.open(file) as src:
            if src.transform == ref_transform:
                print(f"✓ {file.name} ({res_label}) - Transform matches")
            else:
                print(f"✗ {file.name} ({res_label}) - Transform MISMATCH!")
                print(f"  File:      {src.transform}")
                print(f"  Reference: {ref_transform}")
                all_transforms_match = False
    print()

print("=" * 70)
if all_transforms_match:
    print("✓ ALL TRANSFORMS MATCH - PERFECT PIXEL-LEVEL ALIGNMENT!")
    print("\nThis means pixel [i, j] in ANY file represents the")
    print("EXACT SAME geographic location as pixel [i, j] in other files")
    print("of the same resolution. You can safely stack them for CNN!")
else:
    print("✗ SOME TRANSFORMS DON'T MATCH")
    print("Files may be slightly shifted even if bounds look the same")

PIXEL-LEVEL ALIGNMENT CHECK

Reference transforms:
0.1°: | 0.10, 0.00,-25.00|
| 0.00,-0.10, 80.91|
| 0.00, 0.00, 1.00|
0.25°: | 0.25, 0.00,-25.00|
| 0.00,-0.25, 80.91|
| 0.00, 0.00, 1.00|


Checking if transforms match exactly:

### LANDUSE 0.1° ###
✓ clipped_history_2020.tif (0.1°) - Transform matches
✓ clipped_ssp1_26_2030.tif (0.1°) - Transform matches
✓ clipped_ssp1_26_2050.tif (0.1°) - Transform matches
✓ clipped_ssp1_26_2100.tif (0.1°) - Transform matches
✓ clipped_ssp2_45_2030.tif (0.1°) - Transform matches
✓ clipped_ssp2_45_2050.tif (0.1°) - Transform matches
✓ clipped_ssp2_45_2100.tif (0.1°) - Transform matches
✓ clipped_ssp3_70_2030.tif (0.1°) - Transform matches
✓ clipped_ssp3_70_2050.tif (0.1°) - Transform matches
✓ clipped_ssp3_70_2100.tif (0.1°) - Transform matches
✓ clipped_ssp4_34_2030.tif (0.1°) - Transform matches
✓ clipped_ssp4_34_2050.tif (0.1°) - Transform matches
✓ clipped_ssp4_34_2100.tif (0.1°) - Transform matches
✓ clipped_ssp5_85_2030.tif (0.1°) - Transform ma